# 第40章 饼图与环形图（pie）

<!-- module-learning-arc:start -->
> **Matplotlib 模块主线｜第 9 / 12 步：表达不确定性与多视角证据**
>
> **持续应用背景：** 制作经营周会一页报告：把趋势、比较、分布和异常证据组织成有主次、可直接用于会议的静态页面。
>
> **承接上一阶段：** 面积图（fill_between / stackplot）  →  **本章任务：** 饼图与环形图（pie）  →  **下一步：** 误差线与区间图（errorbar）
>
> **大作业连接：** 本章练习将成为《经营周会一页报告》的一部分，最终需要从周会问题出发选择互补图形，完成视觉层级、注释审阅与独立导出。
<!-- module-learning-arc:end -->


## 本章场景

图表怎么选？当你想回答"某个整体当中，哪一块占了大头、哪几块差不多"这类问题时，饼图（pie）就是最直观的选择——它用一个圆把 2 到 5 个互斥类别画成一个个扇区，扇区大小直接反映占比。


## 本章目标

学完本章，你将能够：

- **理解**：理解「饼图与环形图（pie）」的适用场景、数据结构要求，以及它想帮你读出的规律。
- **操作**：能按参数用相应绘图接口画出「饼图与环形图（pie）」，并做必要的美化、注释与导出。
- **迁移**：能换一份真实经营数据，独立画出同类型的「饼图与环形图（pie）」并读出其中的结论。


## 适用场景

**背景引入**：图表怎么选？当你想回答"某个整体当中，哪一块占了大头、哪几块差不多"这类问题时，饼图（pie）就是最直观的选择——它用一个圆把 2 到 5 个互斥类别画成一个个扇区，扇区大小直接反映占比。比起密密麻麻的数字表，把"销售渠道占比""客户满意度构成"这样的占比关系一眼呈现出来，读者几秒钟就能抓住要点，这也是数据分析报告里最常见的一类图。 打个比方：饼图就像切蛋糕——把整体看作一个圆，按占比切成几块；谁占的份额最大、哪几块差不多，用眼睛一扫就明白，所以它最适合回答「占了大头的是谁、各部分比例如何」这种一目了然的占比问题。

展示2至5个互斥类别在同一整体中的比例。


## 图表与参数速查

先用这张表建立本章的方法地图；每一行后面都有对应的独立示例或练习。

| 类别 | 常用方法或写法 | 主要用途 | 需要特别注意 |
| --- | --- | --- | --- |
| 基础图表 | `np.array()`、`plt.subplots()`、`ax.pie()`、`ax.set_title()` | 展示2至5个互斥类别在同一整体中的比例。 | 类别过多 |
| 进阶变体 | `plt.subplots()`、`ax.pie()`、`ax.text()`、`channel_sales.sum()` | 在基础图表上增加分组、注释、布局或交互 | 使用3D效果 |
| 关键参数 | `autopct` | 百分比 | 类别过多 |
| 关键参数 | `startangle` | 起始角 | 使用3D效果 |
| 关键参数 | `wedgeprops` | 扇区样式 | 多个饼图之间比较角度 |
| 关键参数 | `explode` | 轻微突出 | 数据并非同一整体 |


## 准备可复现数据

先完成导入和数据准备，后续单元格只负责一种图表或一种分析动作。


<!-- math-foundation:chapter-40 -->
### 数学推导｜构成图的守恒关系

> 阅读方法：先跟着步骤理解每个量怎样产生，再看最后的可计算形式；不需要脱离业务场景死记公式。

**第 1 步｜先定义同一总体。** $T=\sum_{i=1}^{K}x_i$。

**第 2 步｜每个类别除以同一总体。** $s_i=x_i/T$。

**第 3 步｜验证守恒。** $\sum_i s_i=\sum_i x_i/T=T/T=1$。层级图还要逐个父节点检查

$$
x_{parent}=\sum_{c\in children(parent)}x_c
$$

否则面积虽然能画出来，却不再代表一致的层级构成。

**把上面的关系收束为本章计算式：**

$$
s_i=\frac{x_i}{\sum_jx_j},\qquad \sum_i s_i=1
$$

**符号解释：** $s_i$ 是类别或节点占总体的比例。

**代码对应：** 先聚合并检查 `share.sum()` 接近 1，再传给饼图、矩形树图或旭日图。

**使用边界：** 层级图要求父节点值与子节点口径一致；类别过多时应合并长尾。


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

# 中文字体支持：由平台运行时自动配置
# 说明：Matplotlib 默认字体不含中文字形，中文会显示成方框。
#      本平台在运行每个绘图 cell 前，会自动注册可用的中文字体并设置
#      font.sans-serif / axes.unicode_minus，因此这里不需要手动 import
#      或 addfont，直接使用即可。

# 1️⃣ 数据导入：读取订单数据（指定列类型降低内存、加快分组）
transactions = pd.read_csv(
    "/datasets/uci_online_retail_200k.csv",
    parse_dates=["InvoiceDate"],
    dtype={"Country": "category"},
)
print(f"数据规模：{len(transactions):,} 行 × {transactions.shape[1]} 列")


In [ ]:
# 2️⃣ 特征工程：构造分析所需字段与聚合结果
transactions["amount"] = transactions["Quantity"] * transactions["UnitPrice"]

# 有效订单：数量与单价均为正（退货/取消行不参与月度统计）
completed = transactions.query("Quantity > 0 and UnitPrice > 0")
completed["month"] = (
    completed["InvoiceDate"].dt.to_period("M").astype("string")
)

# 月度聚合：销售额（元）与订单数
monthly_summary = completed.groupby("month").agg(
    sales=("amount", "sum"), orders=("InvoiceNo", "nunique")
)
months = monthly_summary.index.to_numpy()
sales = (monthly_summary["sales"] / 10_000).to_numpy()
orders = monthly_summary["orders"].to_numpy()
profit = sales * 0.18  # 简化假设：利润约为销售额的 18%

# 区域构成：销售额前 4 国，统计「销售 vs 退货」两部分（单位：万元）
top = completed.groupby("Country")["amount"].sum().nlargest(4).index
rows = transactions[transactions["Country"].isin(top)].copy()
rows["flow"] = np.where(rows["Quantity"] > 0, "销售", "退货")
regional = (
    pd.crosstab(
        rows["Country"].astype(str),
        rows["flow"],
        values=rows["amount"].abs(),
        aggfunc="sum",
    )
    / 10_000
).fillna(0)
regions = regional.index.to_numpy()
online = regional["销售"].to_numpy()
offline = regional["退货"].to_numpy()

# 固定随机种子抽样 2000 条，供分布图使用，保证每次运行结果一致
samples = completed["amount"].sample(2_000, random_state=25).to_numpy()
print(f"有效订单：{len(completed):,} 行")


## 例 1｜最小可用图表

先保留必要的编码：位置、颜色或大小。图表标题、坐标轴和单位应能让读者脱离代码理解结果。


In [ ]:
import matplotlib.pyplot as plt

channel_sales = np.array([180, 92, 58])
labels = ["自然流量", "广告", "会员"]
fig, ax = plt.subplots(figsize=(6.5, 5))
ax.pie(
    channel_sales,
    labels=labels,
    autopct="%.1f%%",
    startangle=90,
    colors=["#1a73e8", "#f9ab00", "#188038"],
    wedgeprops={"edgecolor": "white"},
)
ax.set_title("销售渠道占比")
fig.tight_layout()
plt.show()


In [ ]:
# （自动维护）练习上下文快照 1：参考答案将基于此刻的变量运行
_pds_snap_1 = dict(globals())


In [ ]:
try:
    # 请在下方填写代码
    import numpy as np
    import matplotlib.pyplot as plt

    # 基础数据：四个互斥类别，数值即整体中的份额
    pie_labels = ["选项A", "选项B", "选项C", "选项D"]
    # 1) 绘制饼图（改动下方任一参数，运行观察变化）

except Exception as _pds_err:
    print("（练习尚未完成或未填全：", _pds_err, "）")


In [ ]:
# （自动维护）练习上下文快照 2：参考答案将基于此刻的变量运行
_pds_snap_2 = dict(globals())


**练一练**：基础饼图画好了之后，动手改一个参数，看它到底影响什么。先把下面的脚手架跑通，再做这三件事：

1. 把 `startangle` 从 90 改成 0，运行并观察第一个扇区（红色"选项A"）的起始位置发生了什么变化；
2. 把 `autopct="%.1f%%"` 改成 `"%.0f%%"`，对比百分比显示精度从一位小数变成了几位；
3. 在注释行里用一句话记下你的观察，运行自检确认饼图仍然正常绘制。


## 例 2｜进阶变体

在基础图表可读的前提下增加分组、布局、注释或交互。新增编码必须服务于一个明确问题。


In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(6.5, 5))
wedges, texts, autotexts = ax.pie(
    channel_sales,
    labels=labels,
    autopct="%.1f%%",
    startangle=90,
    pctdistance=0.78,
    colors=["#1a73e8", "#f9ab00", "#188038"],
    wedgeprops={"width": 0.38, "edgecolor": "white"},
)
ax.text(
    0,
    0,
    f"总计\n{channel_sales.sum()}",
    ha="center",
    va="center",
    fontsize=14,
    fontweight="bold",
)
ax.set_title("销售渠道构成（环形图）")
fig.tight_layout()
plt.show()


## 参数说明

- autopct：百分比
- startangle：起始角
- wedgeprops：扇区样式
- explode：轻微突出


## 结果解读

主要读取最大、最小和大致占比；精确比较仍应使用柱状图。


## 本章实训：图表只改一个编码

这一组实验专门训练“观察一个结果 → 只改一个变量 → 解释变化”。先运行第一个代码单元格，再运行第二个。


In [ ]:
import matplotlib.pyplot as plt

_demo_months = ["1月", "2月", "3月", "4月"]
_demo_sales = [120, 150, 138, 190]
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(_demo_months, _demo_sales, marker="o")
ax.set_title("月度销售额")
ax.set_ylabel("销售额（万元）")
ax.grid(alpha=0.25)
plt.show()


### 第一个结果怎么读

标题、坐标轴和单位让读者知道图表回答什么问题。没有这些文字，图形即使画出来也不完整。

请记录：输入是什么、输出是什么、输出支持了哪一个结论。


In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.bar(months, sales, color="#2563EB")
ax.axhline(
    sum(sales) / len(sales), color="#DC2626", linestyle="--", label="平均值"
)
ax.set_title("月度销售额与平均值")
ax.set_ylabel("销售额（万元）")
ax.legend()
plt.show()


### 第二个结果怎么读

第二个实验把折线改成柱状图，并增加平均线。请说明：哪种图更适合看趋势，哪种图更适合比较单月差异？

迁移任务：把一个输入值、一个字段或一个图表参数换成自己的例子，再用一句话解释变化。


## 错误恢复：图表能画出但读不懂怎么办

真实数据和真实代码都会出问题。本节先观察问题，再用一个明确的检查或修复步骤恢复运行。


In [ ]:
import matplotlib.pyplot as plt

_demo_months = ["1月", "2月", "3月"]
_demo_sales = [120, 150, 138]
fig, ax = plt.subplots(figsize=(6, 3))
ax.plot(_demo_months, _demo_sales, marker="o")
ax.set_title("月度销售额")
ax.set_xlabel("月份")
ax.set_ylabel("销售额（万元）")
ax.grid(alpha=0.25)
plt.show()


### 错误恢复步骤

1. 先看错误类型、字段或数据形状。
2. 判断问题发生在输入、处理中间结果还是输出。
3. 修复后重新检查结果，而不是只让代码不报错。

图形没有报错不等于结果可用。遇到“看不懂”的图，优先补标题、坐标轴、单位和关键参照线。

迁移任务：把示例中的输入换成一组会触发问题的数据，并记录你的修复规则。


## 易错点提醒

- 类别过多
- 使用3D效果
- 多个饼图之间比较角度
- 数据并非同一整体


## 练习与作业

请使用同一份数据完成下面任务，并说明你选择该图表的原因。完成后补充：图表回答了什么问题、最重要的视觉信号是什么、还有哪些信息无法从图中得出。


## 独立迁移练习

复制最接近的示例，只修改一种视觉编码，并说明阅读任务如何变化。

先在下面单元格完成自己的版本；需要参考时再回看紧邻的示例或参考实现。


In [ ]:
try:
    # 独立迁移练习：把饼图升级为环形图(donut)，让中间可放文字
    # 【目标】用 wedgeprops 挖空圆心，把简单的饼图改成更现代的环形图。
    import matplotlib.pyplot as plt

    # 起点示例(已可运行)：加 width 参数让扇区变成环状，中间留白。
    channel_sales = np.array([180, 92, 58])
    labels = ["自然流量", "广告", "会员"]
    fig, ax = plt.subplots(figsize=(6.5, 5))
    ax.pie(
        channel_sales,
        labels=labels,
        autopct="%.1f%%",
        startangle=90,
        colors=["#1a73e8", "#f9ab00", "#188038"],
        wedgeprops={"edgecolor": "white", "width": 0.45},
    )
    ax.set_title("销售渠道占比（环形）")
    fig.tight_layout()
    plt.show()

    # ---- 反思记录：改成环形后，圆心留白提供了什么 ----
    change_note = "待填写"
    expected_change = "待填写"
    observed_change = "运行后填写"
    print(f"改动：{change_note}")
    print(f"预期：{expected_change}")
    print(f"观察：{observed_change}")

except Exception as _pds_err:
    print("（练习尚未完成或未填全：", _pds_err, "）")


## 小结

在类别很少且总和具有明确整体含义时使用饼图或环形图表达占比。


### 你已经掌握

- 判断饼图与环形图（pie）的适用场景
- 准备与图表匹配的数据结构
- 从基础图表扩展到分组、注释或交互变体
- 按照业务问题解读图表并说明结论边界


### 关键参数

| 参数 | 作用 |
| --- | --- |
| `autopct` | 百分比 |
| `startangle` | 起始角 |
| `wedgeprops` | 扇区样式 |
| `explode` | 轻微突出 |


### 需要注意

- 类别过多
- 使用3D效果
- 多个饼图之间比较角度
- 数据并非同一整体


## 参考答案


### 本章练习


In [ ]:
# 恢复练习 1 时的变量上下文（后面的示例覆盖过这些名字）
globals().update(_pds_snap_1)


In [ ]:
# ===== 完整解答 =====
# 关键改动：startangle=90，autopct 改为 "%.0f%%"（0 位小数），并对比一位小数
import numpy as np
import matplotlib.pyplot as plt

scores = np.array([400, 300, 200, 100])
pie_labels = ["选项A", "选项B", "选项C", "选项D"]

fig, ax = plt.subplots(figsize=(6.5, 5))
ax.pie(
    scores,
    labels=pie_labels,
    autopct="%.0f%%",  # 改为 0 位小数：12、30、50、8
    startangle=90,  # 保持 90（红色"选项A"从正 12 点方向开始）
    colors=["#1a73e8", "#f9ab00", "#188038", "#d93025"],
    wedgeprops={"edgecolor": "white"},
)
ax.set_title("选项分布")
fig.tight_layout()
plt.show()


### 本章练习


In [ ]:
# 恢复练习 2 时的变量上下文（后面的示例覆盖过这些名字）
globals().update(_pds_snap_2)


In [ ]:
import matplotlib.pyplot as plt

satisfaction = np.array([72, 20, 8])
fig, ax = plt.subplots(figsize=(6.5, 5))
ax.pie(
    satisfaction,
    labels=["满意", "一般", "不满意"],
    autopct="%1.0f%%",
    startangle=90,
    colors=["#188038", "#f9ab00", "#d93025"],
    wedgeprops={"width": 0.42, "edgecolor": "white"},
)
ax.set_title("客户满意度构成")
fig.tight_layout()
plt.show()
